# Data Profiling

## 1. Objective
The goal of this notebook is to profile the raw Assorted Medicine Dataset of Bangladesh. 
We will load the raw CSV files, analyze missing values, duplicates, and column types to understand the data before cleaning.

In [1]:
# 2. Imports
import pandas as pd
import os
from pathlib import Path

In [2]:
# 3. Configuration / Paths
# Using pathlib to make paths relative to the project root, ensuring reproducibility across machines.
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Ensure reports directory exists
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# 4. Load & Inspect Data
# Define the files to be profiled from the raw directory.
files = [
    'medicine.csv',
    'generic.csv',
    'manufacturer.csv',
    'dosage form.csv',
    'drug class.csv',
    'indication.csv'
]

profiling_data = []

for f in files:
    filepath = RAW_DIR / f
    if not filepath.exists():
        print(f"File {f} not found.")
        continue
    
    # Load the raw dataset without modifying it.
    # Keeping raw data untouched allows the cleaning process
    # to be reproduced from the original source at any time.
    df = pd.read_csv(filepath, low_memory=False)
    
    # Basic metrics
    num_rows = len(df)
    num_cols = len(df.columns)
    dup_rows = df.duplicated().sum()
    
    print(f"Profiling {f} ({num_rows} rows, {num_cols} columns, {dup_rows} duplicates)")
    
    # Analyze each column
    for col in df.columns:
        col_data = df[col]
        missing_count = col_data.isna().sum()
        missing_pct = (missing_count / num_rows) * 100 if num_rows > 0 else 0
        unique_count = col_data.nunique(dropna=False)
        dtype = str(col_data.dtype)
        
        # Categorical, numeric, text potential
        potential_id = "Yes" if unique_count == num_rows and num_rows > 0 else "No"
        potential_cat = "Yes" if 1 < unique_count < 100 else "No"
        potential_num = "Yes" if pd.api.types.is_numeric_dtype(col_data) else "No"
        potential_text = "Yes" if pd.api.types.is_string_dtype(col_data) and unique_count > 100 else "No"
        
        min_val = col_data.min() if potential_num == "Yes" else None
        max_val = col_data.max() if potential_num == "Yes" else None
        
        sample_vals = col_data.dropna().unique()[:3]
        sample_vals_str = ", ".join([str(x) for x in sample_vals])
        
        # Text specific checks
        empty_strings = 0
        leading_trailing_ws = 0
        suspicious_nulls = 0
        
        if potential_text == "Yes" or potential_cat == "Yes" or dtype == 'object':
            # Count empty strings
            empty_strings = (col_data == "").sum()
            
            # Check leading/trailing whitespace
            str_col = col_data.dropna().astype(str)
            leading_trailing_ws = str_col.str.contains(r'^\s+|\s+$', regex=True).sum()
            
            # Check suspicious null strings
            suspicious_vals = ["N/A", "NA", "null", "None", "-", "NaN", "nan"]
            suspicious_nulls = str_col.str.strip().isin(suspicious_vals).sum()
            
        profiling_data.append({
            'file': f,
            'column': col,
            'rows': num_rows,
            'dtype': dtype,
            'missing': missing_count,
            'missing_pct': missing_pct,
            'unique': unique_count,
            'potential_id': potential_id,
            'potential_cat': potential_cat,
            'potential_num': potential_num,
            'potential_text': potential_text,
            'min': min_val,
            'max': max_val,
            'empty_strings': empty_strings,
            'leading_trailing_ws': leading_trailing_ws,
            'suspicious_nulls': suspicious_nulls,
            'samples': sample_vals_str
        })

Profiling medicine.csv (21714 rows, 10 columns, 0 duplicates)
Profiling generic.csv (1711 rows, 22 columns, 0 duplicates)
Profiling manufacturer.csv (240 rows, 5 columns, 0 duplicates)
Profiling dosage form.csv (113 rows, 4 columns, 0 duplicates)
Profiling drug class.csv (453 rows, 4 columns, 0 duplicates)
Profiling indication.csv (2043 rows, 4 columns, 0 duplicates)


In [4]:
# 5. Save Output
# Save the profiling report to the reports folder
profile_df = pd.DataFrame(profiling_data)
profile_df.to_csv(REPORTS_DIR / 'profiling_report.csv', index=False)
print("Profiling completed and saved.")

# 6. Summary
# This output serves as a reference point for the data cleaning phase.
profile_df.head()

Profiling completed and saved.


,file,column,rows,dtype,missing,missing_pct,unique,potential_id,potential_cat,potential_num,potential_text,min,max,empty_strings,leading_trailing_ws,suspicious_nulls,samples
0,medicine.csv,brand id,21714,int64,0,0.0,21714,Yes,No,Yes,No,1.0,33198.0,0,0,0,"4077, 4006, 6174"
1,medicine.csv,brand name,21714,str,0,0.0,13934,No,No,No,Yes,NaN,NaN,0,0,0,"A-Cold, A-Cof, A-Clox"
2,medicine.csv,type,21714,str,0,0.0,2,No,Yes,No,No,NaN,NaN,0,0,0,"allopathic, herbal"
3,medicine.csv,slug,21714,str,0,0.0,21591,No,No,No,Yes,NaN,NaN,0,0,0,"a-coldsyrup4-mg5-ml, a-cofsyrup10-mg30-mg125-m..."
4,medicine.csv,dosage form,21714,str,0,0.0,112,No,No,No,Yes,NaN,NaN,0,0,0,"Syrup, Injection, Powder for Suspension"
